In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

# 경로 설정
target_dir = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Order-CIV1/de"

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")



In [ ]:
# ① pt_id 치환
print("\n① pt_id 치환 시작")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
            
        print(df.head(3))
        
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pt_id 오류] {file} → {e}")

In [ ]:
# ② hos_dt ±7일 ±30분
print("\n② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'hos_dt' in df.columns:
            df['hos_dt'] = pd.to_datetime(df['hos_dt'], errors='coerce')
            df['hos_dt'] = df['hos_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")



In [ ]:
# med_start_dt 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        
        if 'med_start_dt' in df.columns:

            df['med_start_dt'] = pd.to_datetime(df['med_start_dt'], errors = 'coerce')

            df['med_start_dt'] = df['med_start_dt'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(df['pt_id'].map(time_noise_map).fillna(0).astype(int), unit = "m")

            

            print(df.head(3))

        
            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"med_start_dt 완료: {file}")
      

    except Exception as e:

        print(f"[med_start_dt 오류] {file} -> {e}")

In [ ]:
# med_end_dt 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        
        if 'med_end_dt' in df.columns:

            df['med_end_dt'] = pd.to_datetime(df['med_end_dt'], errors = 'coerce')

            df['med_end_dt'] = df['med_end_dt'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(df['pt_id'].map(time_noise_map).fillna(0).astype(int), unit = "m")

            

            print(df.head(3))

        
            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"med_end_dt 완료: {file}")
      

    except Exception as e:

        print(f"[med_end_dt 오류] {file} -> {e}")

In [ ]:
# med_weight_value 처리
import numpy as np

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')

        if 'med_weight_value' in df.columns:
            df['med_weight_value'] = pd.to_numeric(df['med_weight_value'], errors='coerce')

            # 2. 상하위 1% 경계값 계산
            lower_bound = df['med_weight_value'].quantile(0.01)
            upper_bound = df['med_weight_value'].quantile(0.99)

            # 3. 상하위 1% 바깥값은 경계값으로 대체 (clip)
            df['med_weight_value'] = df['med_weight_value'].clip(lower=lower_bound, upper=upper_bound)

            # 소수점 제거
            df['med_weight_value'] = df['med_weight_value'].apply(
            lambda x: np.floor(x) if pd.notna(x) else np.nan
            )
            
            print(df['med_weight_value'])
            # 저장
            df.to_csv(file, index=False, encoding='utf-8')
            print(f"✅ med_weight_value 처리 완료: {file}")

    except Exception as e:
        print(f"[❗ med_weight_value 오류] {file} → {e}")

In [ ]:
# order_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'order_date' in df.columns:

            df['order_date'] = pd.to_datetime(df['order_date'], errors = 'coerce')

            

            df['order_date'] = df['order_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"order_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[order_date 오류] {file} -> {e}")

In [ ]:
# adm_dt 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'adm_dt' in df.columns:

            df['adm_dt'] = pd.to_datetime(df['adm_dt'], errors = 'coerce')

            

            df['adm_dt'] = df['adm_dt'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(df['pt_id'].map(time_noise_map).fillna(0).astype(int), unit = "m")

            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"adm_dt 완료: {file}")

            

    

    except Exception as e:

        print(f"[adm_dt 오류] {file} -> {e}")

In [ ]:
# surgery_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'surgery_date' in df.columns:

            df['surgery_date'] = pd.to_datetime(df['surgery_date'], errors = 'coerce')

            

            df['surgery_date'] = df['surgery_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"surgery_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[surgery_date 오류] {file} -> {e}")